In [10]:
import json
from pprint import pprint

import requests

from config.challenge import CHALLENGE_INFO
from config.paths import CHALLENGES_DIR
from config.scraping import CHALLENGE_HEADERS

In [11]:
challenge_cat = "Rogue"
challenge_num = 6

In [12]:
chal_base = CHALLENGE_INFO[challenge_cat]["base"]
chal_spec = CHALLENGE_INFO[challenge_cat][challenge_num]

In [ ]:
chal_pref = chal_base.get("pref", "")
chal_suf = chal_spec.get("suf", "")
chal_name = chal_pref + chal_suf
payload = {"date": chal_name}

In [14]:
r = requests.post(
    "https://charlie.topdrivesrecords.com/getCgById",
    headers=CHALLENGE_HEADERS,
    json=payload,
)

In [15]:
scraped_challenge = r.json()

In [16]:
challenge_dict = {}

for i, round_info in enumerate(scraped_challenge["rounds"]):
    r_num = i + 1
    # RQ
    r_dict = {"RQ limit": round_info["rqLimit"], "Tracks": {}, "Restrictions": {}}
    r_dict["RQ range"] = round_info["filter"].get("rqModel", [10, 150])

    # Restrictions
    round_rests = chal_base.get("rest", {}) | chal_spec.get("rest", {})
    for restriction, quant_list in round_rests.items():
        for quant_tup in quant_list:
            if quant_tup[0][0] <= r_num <= quant_tup[0][1]:
                r_dict["Restrictions"][restriction] = quant_tup[1]

    # Races
    for j, race_dict in enumerate(round_info["races"]):
        r_dict["Tracks"][str(j + 1)] = [race_dict["track"], race_dict["time"]]

    challenge_dict[str(r_num)] = r_dict

In [17]:
pprint(challenge_dict)

{'1': {'RQ limit': 70,
       'RQ range': [10, 150],
       'Restrictions': {'tag_Rogue_Agents': 5,
                        'tyres_Standard/tyres_All-surface/tyres_Off-road': 5},
       'Tracks': {'1': ['csMed_a01', 98.82],
                  '2': ['carPark_a01', 55.61],
                  '3': ['mile2_a00', 29.38],
                  '4': ['slalom_a01', 37.91],
                  '5': ['gForce_a01', 29.93]}},
 '10': {'RQ limit': 185,
        'RQ range': [10, 150],
        'Restrictions': {'tag_Rogue_Agents': 5,
                         'tyres_Standard/tyres_All-surface/tyres_Off-road': 5},
        'Tracks': {'1': ['mile4_a00', 15.56],
                   '2': ['mile2_a00', 24.9],
                   '3': ['carPark_a00', 48.39],
                   '4': ['csSmall_a00', 42.5],
                   '5': ['mile1_a00', 41.41]}},
 '2': {'RQ limit': 85,
       'RQ range': [10, 150],
       'Restrictions': {'tag_Rogue_Agents': 5,
                        'tyres_Standard/tyres_All-surface/tyres_Off-road

In [18]:
with open(CHALLENGES_DIR / f"{chal_name}.json", "w", encoding="utf-8") as f:
    json.dump(challenge_dict, f)